In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path 


In [2]:
# Labels for answer choices the data leaves without one: a bare number ('02') or no value label at all (NaN).
# A scale is identified by its labelled choices, in code order; its entry gives the label of every unlabelled code.
# The same scale gets the same labels in any dataset. Labelled choices are never changed.
LABEL_BANK = {
    ("None", "1 child", "2 children"): {
        3: "3 children",
        4: "4 children",
        5: "5 children",
        6: "6 children",
        7: "7 children",
        8: "8 children",
        9: "9 children",
        10: "10 children",
    },
    ("Yes, they definitely should", "No, they definitely should not"): {
        2: "Yes, they should",
        3: "Yes, they probably should",
        4: "Undecided",
        5: "No, they probably should not",
        6: "No, they should not",
    },
    ("You cannot rely on either of them", "You can definitely rely more on close relatives",
     "You can rely equally on both", "You can definitely rely more on close friends"): {
        2: "You can rely more on close relatives",
        3: "You can probably rely more on close relatives",
        5: "You can probably rely more on close friends",
        6: "You can rely more on close friends",
    },
    ("Close relatives should definitely be the most important", "Both should be equally important",
     "Close friends should definitely be the most important"): {
        2: "Close relatives should be the most important",
        3: "Close relatives should probably be the most important",
        5: "Close friends should probably be the most important",
        6: "Close friends should be the most important",
    },
}

BRACKETS = {"[COUNTRY's]": "{country}'s", "[COUNTRY]": "{country}", "[CNTRY]": "{country}",
            "[EU/ USMCA/ etc.]": "{association}"}                       # the data's placeholders -> the bank's

def fill_placeholders(text, fields, where=""):
    """Text with this country's names in place of placeholders: the bank's {country}, {country}'s, {nationality},
    {parliament}, {association}, and the data's own bracket forms (BRACKETS). A 'the' that starts a sentence is
    capitalised. Prints every placeholder filled (where: e.g. the question id), and any it cannot fill."""
    name = fields["country"]
    values = {"{country}'s": name + ("'" if name.endswith("s") else "'s"), **{f"{{{k}}}": v for k, v in fields.items()}}
    for bracket, brace in BRACKETS.items():
        text = text.replace(bracket, brace)
    for placeholder, value in values.items():
        if placeholder in text:
            print(f"{where:<5} {placeholder:<15} ->  {value!r}")
            text = text.replace(placeholder, value)
    text = re.sub(r"(^|[.?!…]\s+)the ", lambda m: m.group(1) + "The ", text)
    for left in re.findall(r"\{[^}]*\}|\[[^\]]*\]", text):
        print(f"{where:<5} {left:<15} ->  UNFILLED (no value for it)")
    return text

def unlabelled(text):
    return not isinstance(text, str) or not re.search(r"[A-Za-z]", text)   # NaN or no letters ('02')

def scale_key(texts):
    return tuple(t for t in texts if not unlabelled(t))                      # labelled choices in code order

def fill_labels(codes, texts):
    fixes = LABEL_BANK.get(scale_key(texts), {})
    return [fixes.get(c, t) if unlabelled(t) else t for c, t in zip(codes, texts)]


COUNTRY_NOTE = re.compile(r"^([A-Z]{2}(?:\s*,\s*[A-Z]{2})*)(?:\s+also)?\s*:\s*")   # 'NL, US: Employers' -> offered only in NL and US

def offered_options(labels, country, verbose=False):
    """{code: text} of every valid (non-negative) option offered in `country`, in code order. An option marked for
    other countries only is dropped; an option marked for `country` keeps its text without the marker; a country note
    after the text ('Never; PL: 1-2 times a year') gives this country's wording if it names this country, else is cut.
    verbose=True prints every option it drops or changes."""
    out = {}
    for k, t in sorted(labels.items()):
        if k < 0:
            continue
        text = re.sub(r"^-?\d+\.\s+", "", t)
        m = COUNTRY_NOTE.match(text)
        if m:
            if country not in re.split(r"\s*,\s*", m[1]):
                if verbose:
                    print(f"  {int(k):>4}  {text!r}  ->  dropped: not offered in {country}")
                continue
            if verbose:
                print(f"  {int(k):>4}  {text!r}  ->  {text[m.end():]!r}")
            text = text[m.end():]
        own = persona_text(text, country)                              # country notes after the text
        if verbose and own != text:
            print(f"  {int(k):>4}  {text!r}  ->  {own!r}")
        out[int(k)] = own
    return out

def persona_text(label, country):
    """Persona value as this country's respondents saw it: '1. Male' -> 'Male'; 'Once a year; US: Once or twice a year'
    -> 'Once or twice a year' in the US, 'Once a year' elsewhere."""
    text = re.sub(r"^-?\d+\.\s+", "", str(label)).strip()
    m = re.fullmatch(r"(.*?)\s*\(([A-Z]{2}(?:\s*,\s*[A-Z]{2})*)\s*:\s*(.*)\)", text)   # a note in brackets: '89 (US: 89 years or older)'
    if m:
        return m[3] if country in re.split(r"\s*,\s*", m[2]) else m[1]
    kept, ours = [], None
    
    for part in re.split(r";|(?<![A-Z]{2}),\s*(?=[A-Z]{2}(?:\s*,\s*[A-Z]{2})*\s*:)", text):
        part = part.strip()
        m = COUNTRY_NOTE.match(part)
        if not m:
            kept.append(part)                          # general wording
        elif country in re.split(r"\s*,\s*", m[1]):
            ours = part[m.end():]                      # this country's own wording
            if re.match(r"(incl|\+|also)", ours, re.I):  # a note that adds to the general wording: 'incl. nearly every week'
                ours = f"{kept[0]}, {ours.lstrip('+ ')}" if kept else ours
    return ours if ours is not None else "; ".join(kept)

def read_missing(path):
    """{variable: codes that are not answers} from GESIS's *_missing.txt: ' v1 (8, 9)', ' DE_v34 (996 THRU 999, 990)',
    ' v46 (LOWEST THRU -4, -1)'."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing-codes file not found: {path}")
    missing = {}
    for line in path.read_text(encoding="latin-1").splitlines():
        match = re.match(r"\s*(\S+)\s*\((.*)\)", line.strip())
        if not match:
            continue
        codes = set()
        for part in match.group(2).upper().replace("LOWEST", "-999").replace("HIGHEST", "99999").split(","):
            if "THRU" in part:
                low, high = part.split("THRU")
                codes |= set(range(int(low), int(high) + 1))
            elif part.strip():
                codes.add(int(float(part)))
        missing[match.group(1)] = codes
    return missing

In [3]:
fn = 'ZA6980_v2-0-0'
data_dir = Path.cwd()/'data/issp2017'
data_fn = data_dir/f'raw/{fn}.dta'
bank_fn = data_dir/'question_bank.csv'
verified_fn = data_dir/'questions_verified.csv'
country = 'US'
long_fn = data_dir/f"issp2017_{country}_long.parquet"
missing = read_missing(data_dir/f'raw/{fn}_missing.txt')
missing["v1"], missing["ATTEND"], missing["VOTE_LE"]


({8, 9}, {97, 98, 99}, {0, 7, 8, 9})

In [4]:
max_options = 15
countries = pd.read_csv(Path.cwd()/'data/issp_countries.csv', keep_default_na=False).set_index("c_alphan")
fields = {**countries.loc[country, ["country", "nationality", "parliament"]].to_dict(),   # this country's names
          "association": "the United States-Mexico-Canada Agreement (USMCA)"}             # 2023 Q38-Q43: data/issp2023/build_dataset.py

bank = pd.read_csv(bank_fn)

raw = pd.read_stata(data_fn, convert_categoricals=False)        # same file, answers as integer codes
labels = pd.io.stata.StataReader(data_fn).variable_labels()     # column -> GESIS label
with pd.io.stata.StataReader(data_fn) as reader:
    label_sets = reader.value_labels()                                   # label-set name -> {code: "N. text"}
    value_labels = {v: label_sets[s] if s in label_sets else {} for v, s in zip(reader._varlist, reader._lbllist)}   # column -> {code: "N. text"}

repeated = [c for c, vl in value_labels.items() if len(set(vl.values())) < len(vl)]   # labels pandas cannot turn into categories (e.g. ISCO08)

wide = pd.read_stata(data_fn, columns=[c for c in raw.columns if c not in repeated])   # as stored; answers shown with GESIS's labels
print(f"raw: {raw.shape}, wide: {wide.shape}, bank: {len(bank)} questions")

raw: (44492, 356), wide: (44492, 351), bank: 67 questions


In [5]:
fixes_fn = data_dir/'label_fixes.csv'                                  # labels the .dta cuts off, completed from the codebook
number = r"^-?\d+\.\s*"                                               # a label's own number prefix ('4. ')
if fixes_fn.exists():
    for fix in pd.read_csv(fixes_fn, keep_default_na=False).itertuples():
        vl = value_labels[fix.variable]
        old = vl[fix.code]
        if re.sub(number, "", old) != re.sub(number, "", fix.data_label):
            raise ValueError(f"Label mismatch for {fix.variable} {fix.code}: got {old!r}, expected {fix.data_label!r}")
        prefix = re.match(number, old)
        new = (prefix.group(0) if prefix else "") + re.sub(number, "", fix.fixed_label)
        value_labels[fix.variable] = {**vl, fix.code: new}
        if fix.variable in wide.columns and old in wide[fix.variable].cat.categories:
            wide[fix.variable] = wide[fix.variable].cat.rename_categories({old: new})   # the answers keep their label
        print(f"{fix.variable} {fix.code}: ...{old[-25:]!r} -> ...{new[-25:]!r}")

v35 4: ...'ful in dealing with peopl' -> ...'ul in dealing with people'
v48 96: ...'most frequently lives wit' -> ...' frequently lives with me'
v49 96: ...'t with the most frequentl' -> ...' frequently lives with me'
v50 96: ...' the most frequently live' -> ...' frequently lives with me'
v51 96: ...'act with the most frequen' -> ...' frequently lives with me'
v52 96: ...'h the most frequently liv' -> ...' frequently lives with me'
v67 2: ...'d that does not allow ent' -> ...'end of obligatory school)'
v67 4: ...' upper secondary programs' -> ...'t or technical formation)'
v67 5: ...'also technical schools at' -> ...'ools at a tertiary level)'
AT_DEGR 7: ...'sity of applied sciences/' -> ...'n, Bachelor at university'
AT_DEGR 8: ...'d sciences and university' -> ...' sciences and university)'
DE_DEGR 12: ...'hhochschule): Master or o' -> ...'): Master or other degree'
FR_DEGR 4: ...'n level 2 or incomplete u' -> ...'ational education level 3'
FR_DEGR 9: ...'on level 5 or Bachelor le' 

In [6]:
# GESIS's non-answer codes that are not negative (e.g. 8 Can't choose, 9 No answer, 0 Not available) -> negative
# (-1000 - code), so every rule below that treats negative codes as non-answers covers them too
missing_files = list(data_dir.glob("raw/*_missing.txt"))
if len(missing_files) != 1:
    raise FileNotFoundError(f"Expected 1 *_missing.txt in {data_dir/'raw'}, found {missing_files}")
missing = read_missing(missing_files[0])
for c, codes in missing.items():
    codes = {k for k in codes if k >= 0}
    if c not in raw.columns or not codes or raw[c].dtype.kind not in "iuf":
        continue
    s = raw[c].astype("float64")
    raw[c] = s.where(~s.isin(codes), -1000 - s)
    value_labels[c] = {(-1000 - int(k) if k in codes else k): t for k, t in value_labels[c].items()}

if "EDULEVEL" not in raw.columns:                                       # up to 2019 education is DEGREE: same role, its own categories
    wide, raw = wide.rename(columns={"DEGREE": "EDULEVEL"}), raw.rename(columns={"DEGREE": "EDULEVEL"})
    value_labels["EDULEVEL"], labels["EDULEVEL"] = value_labels["DEGREE"], labels["DEGREE"]

wide = wide[wide["c_alphan"] == country]
raw = raw.loc[wide.index]                                       # same rows as wide
wide["CASEID"] = wide["CASEID"].astype("int64")                 # stored as a float; shown as 2.022008e+15 otherwise

persona = ["c_alphan", "SEX", "AGE", "EDULEVEL", "RELIGGRP", "ATTEND", "PARTY_LR", "VOTE_LE", "URBRURAL",
           "MARITAL", "MAINSTAT", "UNION", f"{country}_PRTY", f"{country}_INC", f"{country}_REG", f"{country}_ETHN1"]
questions = [c for c in wide.columns if re.fullmatch(r"v\d+", c)]   # v1, v2, ...: the substantive questions

print(f"{country} respondents: {len(wide)}, persona cols: {len(persona)}, raw question cols: {len(questions)}")

US respondents: 1173, persona cols: 16, raw question cols: 67


In [7]:
def all_options(v):
    """{code: text} of every option of question v in this country, in code order: the labelled options offered here
    (offered_options), plus every non-negative code without a label between the labelled ends or given by a respondent
    here (text NaN, filled by LABEL_BANK in the next cell). Codes labelled only for other countries stay out."""
    opts = offered_options(value_labels[v], country)
    given = set(raw.loc[raw[v] >= 0, v].astype(int))                   # codes respondents in this country gave
    labelled = {int(k) for k in value_labels[v] if k >= 0}              # codes with a value label (any country)
    scale = bank.loc[bank["v"] == v, "type"].eq("score").any()           # unlabelled points only exist on ordered scales
    span = set(range(min(opts), max(opts) + 1)) if opts and scale else set()   # every code between the labelled ends
    for c in (span - labelled) | (given - set(opts)):
        opts[c] = float("nan")
    return dict(sorted(opts.items()))

options = {v: all_options(v) for v in questions}                        # v -> {code: text}, every option in this country

wide[questions] = wide[questions].where(raw[questions] >= 0)                          # negative answers -> NaN (that cell only)

questions = [v for v in questions                                              # the one place questions are kept or dropped:
             if v in set(bank.loc[bank["type"] != "exclude", "v"])            #   in the bank and not marked exclude
             and len(options[v]) <= max_options]                              #   at most max_options offered options

bank = bank[bank["v"].isin(questions)]
print(f"Kept {len(questions)} substantive questions (<= {max_options} options, not excluded)")

Kept 66 substantive questions (<= 15 options, not excluded)


In [8]:
# this country's names in place of placeholders, in question texts and option texts
bank["question"] = [fill_placeholders(t, fields, v) for v, t in zip(bank["v"], bank["question"])]
options = {v: {c: fill_placeholders(t, fields, v) if isinstance(t, str) else t for c, t in o.items()} for v, o in options.items()}

wide = wide[["CASEID"] + persona + questions]

wide[questions] = wide[questions].apply(lambda s: s.cat.set_categories(
    [value_labels[s.name][k] if k in value_labels[s.name] else k for k in options[s.name]]))   # categories = every option in this country, in code order

v11   {country}       ->  'the United States'
v14   {country}       ->  'the United States'
v15   {country}       ->  'the United States'
v16   {country}       ->  'the United States'
v36   {country}'s     ->  "the United States'"
v37   {country}       ->  'the United States'


In [9]:
# persona: number prefix removed, this country's wording taken from country notes, negative codes -> NaN
for c in persona[1:]:                                             # c_alphan stays as it is
    vl = value_labels[c]
    values = {}
    for k in sorted(raw[c].dropna().unique()):
        raw_label = int(k) if c == "AGE" and k >= 0 else vl[k]
        before = re.sub(r"^-?\d+\.\s+", "", str(raw_label)).strip()
        if k < 0:
            values[k] = float("nan")
            print(f"{c:<10} {int(k):>6}  {before!r}  ->  NaN  ({int((raw[c] == k).sum())} respondents)")
        else:
            values[k] = persona_text(raw_label, country)
            if values[k] != before:
                print(f"{c:<10} {int(k):>6}  {before!r}  ->  {values[k]!r}")
    wide[c] = [values[k] if pd.notna(k) else float("nan") for k in raw[c]]

AGE         -1999  'No answer'  ->  NaN  (2 respondents)
EDULEVEL    -1009  'No answer'  ->  NaN  (2 respondents)
RELIGGRP    -1099  'No answer'  ->  NaN  (9 respondents)
RELIGGRP        4  'Other Christian; NZ: Christian'  ->  'Other Christian'
ATTEND      -1099  'No answer'  ->  NaN  (8 respondents)
PARTY_LR    -1099  'No answer'  ->  NaN  (64 respondents)
PARTY_LR    -1000  'NAP (Code 0,2,7 in VOTE_LE); NAV (CN,GB,PH,SR,TW)'  ->  NaN  (384 respondents)
VOTE_LE     -1009  'No answer'  ->  NaN  (20 respondents)
VOTE_LE     -1000  'NAP, not eligible to vote at last election; NAV (GB)'  ->  NaN  (59 respondents)
VOTE_LE         2  'No, NZ: incl. was not eligible'  ->  'No'
MARITAL     -1009  'NA, CN: Cohabit'  ->  NaN  (1 respondents)
MAINSTAT    -1099  "No answer; TH: NA incl. don't do anything"  ->  NaN  (1 respondents)
UNION       -1009  'No answer'  ->  NaN  (9 respondents)
US_PRTY     -1099  'No answer'  ->  NaN  (64 respondents)
US_PRTY     -1000  'NAP (Code 0, 2, 7 in VOTE_LE)'  

In [10]:
long = wide.melt(id_vars=["CASEID"]+persona, value_vars=questions, var_name="v", value_name="answer")
long.insert(long.columns.get_loc("v") + 1, "label", [labels[v] for v in long["v"]])
long["answer_code"] = raw[questions].melt()["value"].astype("Int64").to_numpy()   # code from the raw integers, not the label string
long = long.dropna(subset=["answer"])  # drop the NaN cells (negative or missing answers)

long.head(2)

,CASEID,c_alphan,SEX,AGE,EDULEVEL,RELIGGRP,ATTEND,PARTY_LR,VOTE_LE,URBRURAL,...,MAINSTAT,UNION,US_PRTY,US_INC,US_REG,US_ETHN1,v,label,answer,answer_code
0,201700840000002,US,Female,74,Upper secondary (programs that allows entry to...,Catholic,Once a year,"Right, conservative",Yes,The suburbs or outskirts of a big city,...,Retired,"Yes, currently",Republican,30.000-34.999 USD,New England,Italy,v1,"Q1a In circle of friends, relatives and acquai...",No one,4
1,201700840000004,US,Female,63,"Lower level tertiary, first stage (also techni...",Protestant,2 or 3 times a month,NaN,Yes,The suburbs or outskirts of a big city,...,In paid work,"No, never",NaN,170.000 USD or more per year,New England,Ireland,v1,"Q1a In circle of friends, relatives and acquai...",Someone else I know,3


In [11]:
# bank: every option offered in this country, in code order; label fixes shown before -> after
info = bank.set_index("v")
rows = []
for v in questions:
    before = {int(k): re.sub(r"^-?\d+\.\s+", "", t) for k, t in sorted(value_labels[v].items()) if k >= 0}   # as the data has it
    before = dict(sorted((before | {c: float("nan") for c in options[v] if c not in before}).items()))      # + codes without a label
    opts = options[v]                                                                                       # country markers handled
    codes = list(opts)
    after = dict(zip(codes, fill_labels(codes, list(opts.values()))))                                       # unlabelled choices filled
    if any(unlabelled(t) for t in after.values()):                     # no LABEL_BANK entry: the data's numbered form for the whole scale
        after = {c: (t if re.match(rf"{c}\s", t) else f"{c}. {t}") if not unlabelled(t) else f"{c}." for c, t in after.items()}
    rows.append({"v": v,
                 "answer_code": codes,
                 "answer": [value_labels[v].get(k, str(k)) for k in codes],
                 "answer_text": list(after.values())})
    if before == after:
        continue
    print("=" * 100)
    print(f"{v} | {info.loc[v, 'type']} | {info.loc[v, 'data_label']}")
    if any(unlabelled(t) for t in after.values()):
        print("NO ENTRY IN LABEL_BANK FOR THIS SCALE")
    print("-" * 100)
    for c, b in before.items():
        print(f"{c:>4}  {str(b):<45} ->  {after.get(c, f'(dropped: not offered in {country})')}")
print("=" * 100)

bank = bank.merge(pd.DataFrame(rows), on="v")

# long: each person's answer text taken from bank, so the two always agree
text_of = {(r["v"], c): t for _, r in bank.iterrows() for c, t in zip(r["answer_code"], r["answer_text"])}
long["answer_text"] = [text_of[(v, c)] for v, c in zip(long["v"], long["answer_code"])]

# long: the question each answer belongs to, with its type, direction and every option offered
long = long.merge(
    bank[["v", "question", "type", "reversed", "answer_code", "answer_text"]]
        .rename(columns={"answer_code": "option_codes", "answer_text": "option_texts"}),
    on="v", how="left")
long.head(2)

v36 | score | Q12a How much personal trust in: national courts
NO ENTRY IN LABEL_BANK FOR THIS SCALE
----------------------------------------------------------------------------------------------------
   0  00 - No trust at all                          ->  0. 00 - No trust at all
   1  01                                            ->  1.
   2  02                                            ->  2.
   3  03                                            ->  3.
   4  04                                            ->  4.
   5  05                                            ->  5.
   6  06                                            ->  6.
   7  07                                            ->  7.
   8  08                                            ->  8.
   9  09                                            ->  9.
  10  10 - Complete trust                           ->  10 - Complete trust
v37 | score | Q12b How much personal trust in: major private companies
NO ENTRY IN LABEL_BANK FOR THIS SCALE
--

,CASEID,c_alphan,SEX,AGE,EDULEVEL,RELIGGRP,ATTEND,PARTY_LR,VOTE_LE,URBRURAL,...,v,label,answer,answer_code,answer_text,question,type,reversed,option_codes,option_texts
0,201700840000002,US,Female,74,Upper secondary (programs that allows entry to...,Catholic,Once a year,"Right, conservative",Yes,The suburbs or outskirts of a big city,...,v1,"Q1a In circle of friends, relatives and acquai...",No one,4,No one,Here is a list of jobs that people you know ma...,choice,False,"[1, 2, 3, 4]","[Family or relative, Close friend, Someone els..."
1,201700840000004,US,Female,63,"Lower level tertiary, first stage (also techni...",Protestant,2 or 3 times a month,NaN,Yes,The suburbs or outskirts of a big city,...,v1,"Q1a In circle of friends, relatives and acquai...",Someone else I know,3,Someone else I know,Here is a list of jobs that people you know ma...,choice,False,"[1, 2, 3, 4]","[Family or relative, Close friend, Someone els..."


In [12]:
missing = bank[bank["answer_text"].apply(
    lambda texts: any(not isinstance(t, str) or not re.search(r"[A-Za-z]", t) for t in texts))]
missing[["v", "type", "data_label", "answer_code", "answer_text"]]


,v,type,data_label,answer_code,answer_text
35,v36,score,Q12a How much personal trust in: national courts,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[0. 00 - No trust at all, 1., 2., 3., 4., 5., ..."
36,v37,score,Q12b How much personal trust in: major private...,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[0. 00 - No trust at all, 1., 2., 3., 4., 5., ..."
45,v46,score,Q19 In contact with how many people on a typic...,"[1, 2, 3, 4, 5, 6]","[1. 0-4 people, 2., 3., 4., 5., 6. 100 or more]"


In [13]:
for _, r in bank.iterrows():
    v = r["v"]
    # 1. Check that question_bank's data_label matches the Stata variable label
    assert r["data_label"].strip() == labels[v].strip(), f"{v}: {r['data_label']!r} != {labels[v]!r}"

    # 2. Compare {code: text} from question_bank.csv vs {code: text} from the .dta file
    bank_codes = [int(c) for c in str(r["codes"]).split(" | ")]
    bank_opts = [fill_placeholders(t, fields, v) for t in str(r["options"]).split(" | ")]
    bank_map = dict(sorted(zip(bank_codes, bank_opts)))
    dta_map = dict(zip(r["answer_code"], r["answer_text"]))

    if bank_map != dta_map:
        print(f"{v} ({r['data_label']}):")
        for c in sorted(set(bank_map) | set(dta_map)):
            if bank_map.get(c) != dta_map.get(c):
                print(f"  code {c:>2}: bank={bank_map.get(c)!r}  |  dta={dta_map.get(c)!r}")

v36 (Q12a How much personal trust in: national courts):
  code  0: bank='0 No trust at all'  |  dta='0. 00 - No trust at all'
  code  1: bank='1'  |  dta='1.'
  code  2: bank='2'  |  dta='2.'
  code  3: bank='3'  |  dta='3.'
  code  4: bank='4'  |  dta='4.'
  code  5: bank='5'  |  dta='5.'
  code  6: bank='6'  |  dta='6.'
  code  7: bank='7'  |  dta='7.'
  code  8: bank='8'  |  dta='8.'
  code  9: bank='9'  |  dta='9.'
  code 10: bank='10 Complete trust'  |  dta='10 - Complete trust'
v37 (Q12b How much personal trust in: major private companies):
  code  0: bank='0 No trust at all'  |  dta='0. 00 - No trust at all'
  code  1: bank='1'  |  dta='1.'
  code  2: bank='2'  |  dta='2.'
  code  3: bank='3'  |  dta='3.'
  code  4: bank='4'  |  dta='4.'
  code  5: bank='5'  |  dta='5.'
  code  6: bank='6'  |  dta='6.'
  code  7: bank='7'  |  dta='7.'
  code  8: bank='8'  |  dta='8.'
  code  9: bank='9'  |  dta='9.'
  code 10: bank='10 Complete trust'  |  dta='10 - Complete trust'
v46 (Q19 In co

### Check persona

In [14]:
EDULEVEL_MAP = {
    "No formal education": "Less than high school",
    "Primary school (elementary education)": "Less than high school",
    "Lower secondary (secondary completed that does not allow entry to university: end of obligatory school)": "Less than high school",
    "Upper secondary (programs that allows entry to university)": "Upper secondary",
    "Post secondary, non-tertiary (other upper secondary programs toward the labour market or technical formation)": "Short-cycle tertiary",
    "Lower level tertiary, first stage (also technical schools at a tertiary level)": "College graduate/some postgrad",
    "Upper level tertiary (Master, Doctor)": "College graduate/some postgrad",
}
if set(long["EDULEVEL"].dropna().unique()) != set(EDULEVEL_MAP.keys()):
    raise ValueError(f"Unexpected EDULEVEL values: {set(long['EDULEVEL'].dropna().unique()) ^ set(EDULEVEL_MAP.keys())}")
long["EDULEVEL"] = [EDULEVEL_MAP[v] if pd.notna(v) else v for v in long["EDULEVEL"]]
long["EDULEVEL"].value_counts(dropna=False)

EDULEVEL
Upper secondary                   33829
College graduate/some postgrad    21839
Less than high school              7276
Short-cycle tertiary               6448
NaN                                 108
Name: count, dtype: int64

In [15]:
RELIGGRP_MAP = {
    "No religion": "Atheist",
    "Islamic": "Muslim",
    "Protestant": "Protestant",
    "Catholic": "Catholic",
    "Orthodox": "Orthodox",
    "Other Christian": "Other Christian",
    "Jewish": "Jewish",
    "Buddhist": "Buddhist",
    "Hindu": "Hindu",
    "Other Asian Religions": "Other Asian Religions",
    "Other Religions": "Other Religions",
}
if set(long["RELIGGRP"].dropna().unique()) != set(RELIGGRP_MAP.keys()):
    raise ValueError(f"Unexpected RELIGGRP values: {set(long['RELIGGRP'].dropna().unique()) ^ set(RELIGGRP_MAP.keys())}")
long["RELIGGRP"] = [RELIGGRP_MAP[v] if pd.notna(v) else v for v in long["RELIGGRP"]]
long["RELIGGRP"].value_counts(dropna=False)

RELIGGRP
Protestant               35027
Atheist                  16654
Catholic                 13357
Jewish                    1132
Other Religions            771
Buddhist                   654
Other Christian            534
NaN                        489
Muslim                     411
Hindu                      239
Orthodox                   175
Other Asian Religions       57
Name: count, dtype: int64

In [16]:
PARTY_LR_MAP = {
    "Center, liberal": "Liberal",
    "Right, conservative": "Conservative",
    "Other": "Other",
}
if set(long["PARTY_LR"].dropna().unique()) != set(PARTY_LR_MAP.keys()):
    raise ValueError(f"Unexpected PARTY_LR values: {set(long['PARTY_LR'].dropna().unique()) ^ set(PARTY_LR_MAP.keys())}")
long["PARTY_LR"] = [PARTY_LR_MAP[v] if pd.notna(v) else v for v in long["PARTY_LR"]]
long["PARTY_LR"].value_counts(dropna=False)

PARTY_LR
NaN             26471
Liberal         23386
Conservative    17140
Other            2503
Name: count, dtype: int64

In [17]:
US_PRTY_MAP = {
    "Democrat": "Democrat",
    "Republican": "Republican",
    "Other Candidate": "Other candidate",
}
if set(long["US_PRTY"].dropna().unique()) != set(US_PRTY_MAP.keys()):
    raise ValueError(f"Unexpected US_PRTY values: {set(long['US_PRTY'].dropna().unique()) ^ set(US_PRTY_MAP.keys())}")
long["US_PRTY"] = [US_PRTY_MAP[v] if pd.notna(v) else v for v in long["US_PRTY"]]
long["US_PRTY"].value_counts(dropna=False)

US_PRTY
NaN                26471
Democrat           23386
Republican         17140
Other candidate     2503
Name: count, dtype: int64

In [18]:
US_INC_MAP = {
    "Under 1.000 USD per year": "Less than $30,000",
    "1.000-2.999 USD": "Less than $30,000",
    "3.000-3.999 USD": "Less than $30,000",
    "4.000-4.999 USD": "Less than $30,000",
    "5.000-5.999 USD": "Less than $30,000",
    "6.000-6.999 USD": "Less than $30,000",
    "7.000-7.999 USD": "Less than $30,000",
    "8.000-9.999 USD": "Less than $30,000",
    "10.000-12.499 USD": "Less than $30,000",
    "12.500-14.999 USD": "Less than $30,000",
    "15.000-17.499 USD": "Less than $30,000",
    "17.500-19.999 USD": "Less than $30,000",
    "20.000-22.499 USD": "Less than $30,000",
    "22.500-24.999 USD": "Less than $30,000",
    "25.000-29.999 USD": "Less than $30,000",
    "30.000-34.999 USD": "$30,000 - $34,999",
    "35.000-39.999 USD": "$35,000 - $39,999",
    "40.000-49.999 USD": "$40,000 - $49,999",
    "50.000-59.999 USD": "$50,000 - $59,999",
    "60.000-74.999 USD": "$60,000 - $74,999",
    "75.000-89.999 USD": "$75,000 - $89,999",
    "90.000-109.999 USD": "$90,000 - $109,999",
    "110.000-129.999 USD": "$100,000 or more",
    "130.000-149.999 USD": "$100,000 or more",
    "150.000-169.999 USD": "$100,000 or more",
    "170.000 USD or more per year": "$100,000 or more",
}
if set(long["US_INC"].dropna().unique()) != set(US_INC_MAP.keys()):
    raise ValueError(f"Unexpected US_INC values: {set(long['US_INC'].dropna().unique()) ^ set(US_INC_MAP.keys())}")
long["US_INC"] = [US_INC_MAP[v] if pd.notna(v) else v for v in long["US_INC"]]
long["US_INC"].value_counts(dropna=False)

US_INC
Less than $30,000     17733
$100,000 or more      13780
$60,000 - $74,999      6387
NaN                    5578
$75,000 - $89,999      5131
$50,000 - $59,999      5101
$40,000 - $49,999      5055
$90,000 - $109,999     4480
$30,000 - $34,999      3686
$35,000 - $39,999      2569
Name: count, dtype: int64

In [19]:
US_REG_MAP = {
    "New England": "Northeast",
    "Middle Atlantic": "Northeast",
    "South Atlantic": "South",
    "East South Central": "South",
    "West South Central": "South",
    "East North Central": "East North Central",
    "West North Central": "West North Central",
    "Mountain": "Mountain",
    "Pacific": "Pacific",
}
if set(long["US_REG"].dropna().unique()) != set(US_REG_MAP.keys()):
    raise ValueError(f"Unexpected US_REG values: {set(long['US_REG'].dropna().unique()) ^ set(US_REG_MAP.keys())}")
long["US_REG"] = [US_REG_MAP[v] if pd.notna(v) else v for v in long["US_REG"]]
long["US_REG"].value_counts(dropna=False)

US_REG
South                 28304
East North Central    11860
Northeast             10426
Pacific               10156
Mountain               5192
West North Central     3562
Name: count, dtype: int64

In [20]:
ATTEND_MAP = {
    "Several times a week or more often (incl. every day, several times a day)": "Several times a week or more often",
    "Once a week": "Once a week",
    "2 or 3 times a month": "2 or 3 times a month",
    "Once a month": "Once a month",
    "Several times a year": "Several times a year",
    "Once a year": "Once or twice a year",
    "Less frequently than once a year": "Less frequently than once a year",
    "Never": "Never",
}
if set(long["ATTEND"].dropna().unique()) != set(ATTEND_MAP.keys()):
    raise ValueError(f"Unexpected ATTEND values: {set(long['ATTEND'].dropna().unique()) ^ set(ATTEND_MAP.keys())}")
long["ATTEND"] = [ATTEND_MAP[v] if pd.notna(v) else v for v in long["ATTEND"]]
long["ATTEND"].value_counts(dropna=False)

ATTEND
Never                                 21478
Once a week                           11187
Once or twice a year                   9217
2 or 3 times a month                   8340
Several times a year                   7289
Less frequently than once a year       4388
Once a month                           3868
Several times a week or more often     3279
NaN                                     454
Name: count, dtype: int64

In [21]:
MARITAL_MAP = {
    "Married": "Married",
    "Separated from spouse/ civil partner (still legally married/ still legally in a civil partnership)": "Separated from spouse/civil partner (still legally married/still legally in a civil partnership)",
    "Divorced from spouse/ legally separated from civil partner,": "Divorced from spouse/legally separated from civil partner",
    "Widowed/ civil partner died": "Widowed/civil partner died",
    "Never married/ never in a civil partnership, single": "Never married/never in a civil partnership",
}
if set(long["MARITAL"].dropna().unique()) != set(MARITAL_MAP.keys()):
    raise ValueError(f"Unexpected MARITAL values: {set(long['MARITAL'].dropna().unique()) ^ set(MARITAL_MAP.keys())}")
long["MARITAL"] = [MARITAL_MAP[v] if pd.notna(v) else v for v in long["MARITAL"]]
long["MARITAL"].value_counts(dropna=False)

MARITAL
Married                                                                                             29565
Never married/never in a civil partnership                                                          20196
Divorced from spouse/legally separated from civil partner                                           11342
Widowed/civil partner died                                                                           5612
Separated from spouse/civil partner (still legally married/still legally in a civil partnership)     2725
NaN                                                                                                    60
Name: count, dtype: int64

In [22]:
UNION_MAP = {
    "Yes, currently": "Yes",
    "No, never": "No",
}
if set(long["UNION"].dropna().unique()) != set(UNION_MAP.keys()):
    raise ValueError(f"Unexpected UNION values: {set(long['UNION'].dropna().unique()) ^ set(UNION_MAP.keys())}")
long["UNION"] = [UNION_MAP[v] if pd.notna(v) else v for v in long["UNION"]]
long["UNION"].value_counts(dropna=False)

UNION
No     62142
Yes     6893
NaN      465
Name: count, dtype: int64

In [23]:
US_ETHN1_MAP = {
    "Africa": "Other African",
    "French Canada": "Canada (French)",
    "Other Canada": "Canada",
    "England & Wales": "England and Wales",
    "Netherlands": "Netherlands (Dutch / Holland)",
    "Russia": "Russia (USSR)",
    "Non-Span Windies": "Other Caribbean",
    "American Only": "American only",
}
missing_ethn = set(US_ETHN1_MAP.keys()) - set(long["US_ETHN1"].dropna().unique())
if missing_ethn:
    raise ValueError(f"Expected US_ETHN1 keys not found: {missing_ethn}")
long["US_ETHN1"] = [US_ETHN1_MAP[v] if v in US_ETHN1_MAP else v for v in long["US_ETHN1"]]
long["US_ETHN1"].value_counts(dropna=False).head(15)

US_ETHN1
Germany              9434
NaN                  8414
England and Wales    6608
Ireland              6197
Other African        5875
Mexico               4253
Scotland             2562
Italy                2438
American Indian      2134
American only        1908
Poland               1842
France               1780
Other Spanish        1530
Other European       1196
Spain                1131
Name: count, dtype: int64

In [24]:
bad_persona = {}
for c in persona:
    invalid = [
        v for v in long[c].dropna().unique()
        if re.search(r";|\b[A-Z]{2}(?:\s*,\s*[A-Z]{2})*\s*:|^-?\d+\.\s+|\b(No answer|Don't know|Refused|NAP|NAV|Not available|Can't choose)\b", str(v), re.I)
    ]
    if invalid:
        bad_persona[c] = invalid
if set(long["c_alphan"].unique()) != {"US"}:
    bad_persona["c_alphan"] = list(long["c_alphan"].unique())
if set(long["SEX"].dropna().unique()) != {"Male", "Female"}:
    bad_persona["SEX"] = list(long["SEX"].dropna().unique())
if set(long["VOTE_LE"].dropna().unique()) != {"Yes", "No"}:
    bad_persona["VOTE_LE"] = list(long["VOTE_LE"].dropna().unique())
if bad_persona:
    raise ValueError(f"Unnormalized persona values found: {bad_persona}")

# Export verified

In [25]:
# ---- export for training: one row per answer, with the person's persona, the question and every option offered
columns = ["CASEID"] + persona + ["v", "label", "question", "type", 
                                  "reversed", "option_codes", "option_texts",
                                  "answer_code", "answer_text"
                                ]
long[columns].to_parquet(long_fn, index=False)
long_fn


PosixPath('/home/dw/github/decision-models-finetuning/code/data/issp2017/issp2017_US_long.parquet')

In [26]:
bank['options_old'] = bank['options']
bank[['v', 'data_label', 'question', 'options_old', 'answer_code', 'answer_text', 'type', 'reversed']].to_csv(verified_fn,index=False)
